In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from mxlpy import Model, Scipy, Simulator, cartesian_product, make_protocol, scan

from mxlmodels import Simulator, get_ebenhoeh2014


def bdf_integrator(
    rhs: Model, y0: tuple[float, ...], jacobian=None, t0: float = 0.0
) -> Scipy:
    return Scipy(
        rhs=rhs, y0=y0, jacobian=jacobian, t0=t0, method="BDF", atol=1e-8, rtol=1e-8
    )

In [ ]:
_i_flash = 5000.0  # uE
_i_dark = 1e-7  # uE
_i_light = 100  # uE

_t_flash = 0.8  # s
_t_ambient = 90  # s

protocol = make_protocol(
    [
        *[
            (_t_flash, {"PPFD": _i_flash}),
            (_t_ambient - _t_flash, {"PPFD": _i_dark}),
        ]
        * 3,
        *[
            (_t_flash, {"PPFD": _i_flash}),
            (_t_ambient - _t_flash, {"PPFD": _i_light}),
        ]
        * 7,
        *[
            (_t_flash, {"PPFD": _i_flash}),
            (_t_ambient - _t_flash, {"PPFD": _i_dark}),
        ]
        * 10,
    ]
)

m = get_ebenhoeh2014()
p0 = m.get_parameter_values()
m.update_variables(
    {
        "Plastoquinone (oxidised)": p0["PQ_tot"] / 2,
        "Plastocyanine (oxidised)": p0["PC_tot"] / 2,
        "Ferredoxine (oxidised)": p0["Fd*"] / 2,
        "ATP": p0["A*P"] / 3,
        "NADPH": p0["NADP*"] / 2,
        "Light-harvesting complex": 0.9,
    }
)

res = (
    Simulator(m, integrator=bdf_integrator)
    .simulate_protocol(protocol, time_points_per_step=40)
    .get_result()
    .unwrap_or_err()
    .get_combined()
)

flou = res["Fluo"].to_numpy()
norm_flou = flou / flou.max()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(res.index, norm_flou, color="0.55", lw=0.8, label="Simulation")
ax.set(
    xlabel="time [s]",
    ylabel="fluorescence",
    title="Fig. 2a — light-induced state transitions",
    xlim=(0, 1800),
    ylim=(0, 1.05),
)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# anoxic shift
_dT = 120
_t_flash = 0.8
_i_flash = 5000.0

O2ext = p0["O2 (dissolved)_lumen"]
kNDH = p0["kf_ndh"]
_i_light = 0


protocol = make_protocol(
    [
        # 4 oxic cycles (t = 0 ... 480)
        *[
            (
                _t_flash,
                {"PPFD": _i_flash, "O2 (dissolved)_lumen": O2ext, "kf_ndh": 0.0},
            ),
            (
                _dT - _t_flash,
                {"PPFD": _i_light, "O2 (dissolved)_lumen": O2ext, "kf_ndh": 0.0},
            ),
        ]
        * 4,
        # 11 anoxic cycles (t = 480 ... 1800)
        *[
            (_t_flash, {"PPFD": _i_flash, "O2 (dissolved)_lumen": 0.0, "kf_ndh": kNDH}),
            (
                _dT - _t_flash,
                {"PPFD": _i_light, "O2 (dissolved)_lumen": 0.0, "kf_ndh": kNDH},
            ),
        ]
        * 11,
        # 15 oxic cycles (t = 1800 ... 3600)
        *[
            (
                _t_flash,
                {"PPFD": _i_flash, "O2 (dissolved)_lumen": O2ext, "kf_ndh": 0.0},
            ),
            (
                _dT - _t_flash,
                {"PPFD": _i_light, "O2 (dissolved)_lumen": O2ext, "kf_ndh": 0.0},
            ),
        ]
        * 15,
    ]
)

m.update_variables(
    {
        "Plastoquinone (oxidised)": p0["PQ_tot"],
        "Plastocyanine (oxidised)": 0.0202,
        "Ferredoxine (oxidised)": 5.0,
        "ATP": 0.0,
        "NADPH": 0.0,
        "Light-harvesting complex": 0.9,
    }
)

res = (
    Simulator(m, integrator=bdf_integrator)
    .simulate_protocol(protocol, time_points_per_step=40)
    .get_result()
    .unwrap_or_err()
    .get_combined()
)

flou = res["Fluo"].to_numpy()
norm_flou = flou / flou.max()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(res.index, norm_flou, color="0.55", lw=0.8, label="Simulation (MxlPy)")
ax.set(
    xlabel="time [s]",
    ylabel="fluorescence",
    title="Fig. 2b — anoxia-induced state transitions",
    xlim=(0, 3200),
    ylim=(0, 1.05),
)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
m = get_ebenhoeh2014()
# figure 3

antenna_crosssection = np.linspace(0, 1, 20)
ligth_intensity = np.linspace(25, 500, 20)

grid = cartesian_product(
    {
        "staticAntII": antenna_crosssection,
        "PPFD": ligth_intensity,
    }
)

m.update_parameters(
    {
        "staticAntI": 0.0,
        "kStt7": 0.0,
        "kPph1": 0.0,
    }
)
m.update_variable("Light-harvesting complex", 0.0)


s = scan.steady_state(m, to_scan=grid, integrator=bdf_integrator)

reduced_PQ = (
    s.variables["PQ_red/tot"]
    .unstack("staticAntII")  # Zeilen=PPFD, Spalten=staticAntII
    .reindex(index=ligth_intensity, columns=antenna_crosssection)
    .to_numpy()
)

fig = plt.figure(figsize=(10, 12))
ax = fig.add_subplot(111, projection="3d")

X, Y = np.meshgrid(antenna_crosssection, ligth_intensity)
surf = ax.plot_surface(
    X,
    Y,
    reduced_PQ,
    cmap=cm.jet,
    vmin=0,
    vmax=1,
    rstride=1,
    cstride=1,
    edgecolor="k",
    linewidth=0.2,
    antialiased=True,
)
ax.set_xlabel("relative antenna cross section of PSII")
ax.set_ylabel("light intensity")
ax.set_zlabel("fraction of reduced PQ")
ax.set_title("Fig. 3 — steady-state PQ redox state")
ax.set_zlim(0, 1)
ax.view_init(elev=22, azim=-128)
fig.colorbar(surf, ax=ax, shrink=0.6, label="PQ red / PQ tot")
fig.tight_layout()
plt.show()